# 17 — Validate saved extraction evidence against its sources

This notebook inspects **the original four-label run referenced by notebook 13's committed transformer record**. Later experience-enabled runs are inventoried separately; they are not pooled into or substituted for this historical run.

There are two different questions:

- **Source integrity:** Is each record an exact slice of the original document field, with valid coordinates, provenance, score and coverage?
- **Semantic accuracy:** Is its entity label correct, and which entities were missed?

Source agreement answers the first question only. A phrase really appearing in a CV does not prove that it is a DEGREE, TITLE or skill. Djinni Fit labels are not span annotations. This notebook performs no model inference, training, threshold tuning or new annotation.

Scope follows the approved [validation plan](../plan/2026-10-14-entity-extraction-validation/README.md). Real examples are short, contact-redacted inspection excerpts; original source text and offsets are unchanged.

In [1]:
import json
from pathlib import Path
from candidate_screener.config import PROCESSED
import pandas as pd
from candidate_screener.annotation.redact import redact_text, assert_clean
from candidate_screener.extraction.validation import (
    inventory_runs, audit_saved_run, examples, inventory_gold,
    comparison_examples, suspicious_examples,
)

# None means the original run in output/extraction/transformer-metrics.json.
# Set an explicit run ID only to inspect a different population separately.
RUN_ID = None

# Text displays are bounded; none prints a complete document.
def show_summary(value):
    rendered = json.dumps(value, indent=2, default=str, ensure_ascii=False)
    print(rendered if len(rendered) <= 1900 else rendered[:1850] + "\n[output truncated]")


## Saved-run inventory

Compare run IDs, label prompts, completion and population sizes before reading results. Different label inventories or samples describe different experiments. Only the selected run is audited below. Up to five saved runs are shown per inventory preview.

In [2]:
runs = inventory_runs()
print("Saved runs inventoried:", len(runs))
print(runs.iloc[:5].to_json(orient="records", force_ascii=False))


Saved runs inventoried: 8
[{"run_id":"15b9823ab25a15b4dbef-20261009T145542008560Z","status":"complete","requested_documents":200,"completed_documents":200,"labels":"DEGREE, EXPERIENCE_YEARS, KNOWLEDGE, SKILL, TITLE","threshold":0.5},{"run_id":"17996933988a2e768760-20261009T135259563027Z","status":"complete","requested_documents":10,"completed_documents":10,"labels":"DEGREE, EXPERIENCE_YEARS, KNOWLEDGE, SKILL, TITLE","threshold":0.5},{"run_id":"221a773317b7ab8f052f-20261011T045828300384Z","status":"complete","requested_documents":10,"completed_documents":10,"labels":"DEGREE, EXPERIENCE_YEARS, KNOWLEDGE, SKILL, TITLE","threshold":0.5},{"run_id":"2c591cc8bff7fc9e7189-20261005T164947399919Z","status":"complete","requested_documents":200,"completed_documents":200,"labels":"DEGREE, KNOWLEDGE, SKILL, TITLE","threshold":0.5},{"run_id":"3a7d35be6eaa4388e2f5-20261011T044909607182Z","status":"complete","requested_documents":200,"completed_documents":200,"labels":"DEGREE, EXPERIENCE_YEARS, KNOWLED

In [3]:
print(runs.iloc[5:10].to_json(orient="records", force_ascii=False))

[{"run_id":"4fde336ec50802f910d2-20261009T161236977778Z","status":"complete","requested_documents":200,"completed_documents":200,"labels":"DEGREE, EXPERIENCE_YEARS, KNOWLEDGE, SKILL, TITLE","threshold":0.5},{"run_id":"c6d4177ff3b3bdd07c47-20261009T160033414070Z","status":"complete","requested_documents":10,"completed_documents":10,"labels":"DEGREE, EXPERIENCE_YEARS, KNOWLEDGE, SKILL, TITLE","threshold":0.5},{"run_id":"cd9f147acc576d4bab2d-20261005T164350083590Z","status":"complete","requested_documents":10,"completed_documents":10,"labels":"DEGREE, KNOWLEDGE, SKILL, TITLE","threshold":0.5}]


## Source-integrity audit

The helper reconstructs the run's exact source fields from its saved sample manifest. It checks document/field identity, bounds and original surface slices, duplicate records, finite bounded scores, source/sample hashes and field coverage. Failures are reported explicitly; neither text nor predictions are silently repaired or regenerated.

Historical code digests are retained rather than required to equal current code. Read the source/sample checks and mismatch counts; code age alone does not establish that historical spans changed.

In [4]:
audit = audit_saved_run(run_id=RUN_ID)
show_summary(audit["report"])

report_directory = PROCESSED / "extraction" / "validation"
report_directory.mkdir(parents=True, exist_ok=True)
report_path = report_directory / (audit["report"]["run_id"] + ".json")
report_path.write_text(json.dumps(audit["report"], indent=2, sort_keys=True), encoding="utf-8")


{
  "passed": true,
  "failures": {},
  "entities_checked": 1832,
  "documents_checked": 200,
  "fields_checked": 600,
  "scope": "Source traceability and operational integrity; not entity-label accuracy or missing-entity recall",
  "run_id": "2c591cc8bff7fc9e7189-20261005T164947399919Z",
  "labels": [
    "DEGREE",
    "KNOWLEDGE",
    "SKILL",
    "TITLE"
  ],
  "threshold": 0.5,
  "by_label": {
    "KNOWLEDGE": 1330,
    "TITLE": 351,
    "SKILL": 109,
    "DEGREE": 42
  },
  "by_document_type": {
    "cv": 1077,
    "jd": 755
  },
  "historical_code_policy": "Recorded historical digests retained; current-code equality is not required"
}


648

In [5]:
provenance = audit["provenance"]
show_summary({"label_prompts": provenance["labels"], "extractor_version": provenance.get("transformer_version"), "checkpoint": {key: provenance["model"][key] for key in ("model_id", "revision")}, "source_sha256": provenance["input_sha256"], "partition_sha256": provenance["partition_sha256"]})


{
  "label_prompts": {
    "academic degree or qualification": "DEGREE",
    "job title or occupation": "TITLE",
    "professional skill or competency": "SKILL",
    "technology, software tool or programming language": "KNOWLEDGE"
  },
  "extractor_version": "gliner-adapter-v1",
  "checkpoint": {
    "model_id": "urchade/gliner_small-v2.1",
    "revision": "4e091416cf7c3481db542c2a3d26156916f3a47f"
  },
  "source_sha256": {
    "djinni-cv/default-train-0000.parquet": "a7b1378e0b231507c377659bf2c80fa4ba60c4e31e1107b13425d103879ecc3b",
    "djinni-jd/default-train-0000.parquet": "71dddaf0d9660f34ddf946953b67f54f1e7d60e22dcf9a4b8139b6378b18aab9"
  },
  "partition_sha256": "e8b7951db4ef346390e03166b356a10658beb4a65a720571b4360bf4c46c1b85"
}


## Real examples by label and document type

Each example displays an extracted surface next to bounded source context, with source field, label, raw uncalibrated score and original offsets. Contact redaction runs **inside the notebook** before display. Redaction is not a general name-removal guarantee. Display strings are separate from the immutable source text used by the audit.

The rows are deterministic inspection examples, not gold judgements or a representative estimate of error rates. Three rows per label/source keep the text comfortably below the five-row/2,000-character limit.

### KNOWLEDGE — CV

In [6]:
preview = examples(audit, label="KNOWLEDGE", doc_type="cv", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "KNOWLEDGE cv surfaces")
assert_clean(preview["context"], "KNOWLEDGE cv bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"IT","label":"KNOWLEDGE","score":0.8032386303,"doc_type":"cv","field":"Looking For","start":52,"end":54,"context":"resting projects in IT with good career gr"},{"surface":"marketing automation","label":"KNOWLEDGE","score":0.6583743095,"doc_type":"cv","field":"Moreinfo","start":120,"end":140,"context":" marketing systems (marketing automation, CRM project with 2"},{"surface":"JIRA","label":"KNOWLEDGE","score":0.595909059,"doc_type":"cv","field":"Moreinfo","start":348,"end":352,"context":"nd task management (JIRA, Wrike & Redmine);\r"}]


### KNOWLEDGE — JD

In [7]:
preview = examples(audit, label="KNOWLEDGE", doc_type="jd", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "KNOWLEDGE jd surfaces")
assert_clean(preview["context"], "KNOWLEDGE jd bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Laravel framework","label":"KNOWLEDGE","score":0.5454676151,"doc_type":"jd","field":"Long Description","start":1297,"end":1314,"context":"nderstanding of the Laravel framework\r\n- Understanding of"},{"surface":"PostgreSQL","label":"KNOWLEDGE","score":0.6279338598,"doc_type":"jd","field":"Long Description","start":1335,"end":1345,"context":"\n- Understanding of PostgreSQL and its core princi"},{"surface":"PHPUnit","label":"KNOWLEDGE","score":0.6222729087,"doc_type":"jd","field":"Long Description","start":1701,"end":1708,"context":"by automated tests (PHPUnit)\r\n- Familiarity wit"}]


### SKILL — CV

In [8]:
preview = examples(audit, label="SKILL", doc_type="cv", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "SKILL cv surfaces")
assert_clean(preview["context"], "SKILL cv bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"proven leadership and organization skills","label":"SKILL","score":0.6281690001,"doc_type":"cv","field":"Moreinfo","start":596,"end":637,"context":"\r\n\r\nMain skills:\r\n- proven leadership and organization skills\r\n- teamwork and col"},{"surface":"teamwork and collaboration","label":"SKILL","score":0.6447965503,"doc_type":"cv","field":"Moreinfo","start":641,"end":667,"context":"anization skills\r\n- teamwork and collaboration \r\n- system thinking"},{"surface":"system thinking","label":"SKILL","score":0.6321334243,"doc_type":"cv","field":"Moreinfo","start":672,"end":687,"context":"d collaboration \r\n- system thinking"}]


### SKILL — JD

In [9]:
preview = examples(audit, label="SKILL", doc_type="jd", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "SKILL jd surfaces")
assert_clean(preview["context"], "SKILL jd bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Team player with excellent communication and interpersonal skills","label":"SKILL","score":0.5095896721,"doc_type":"jd","field":"Long Description","start":820,"end":885,"context":"wer BI is a plus\r\n- Team player with excellent communication and interpersonal skills\r\n- Strong analytica"},{"surface":"Strong analytical and problem-solving skills","label":"SKILL","score":0.706563592,"doc_type":"jd","field":"Long Description","start":889,"end":933,"context":"rpersonal skills\r\n- Strong analytical and problem-solving skills\r\n- Cross cultural e"},{"surface":"Good English skills","label":"SKILL","score":0.6374449134,"doc_type":"jd","field":"Long Description","start":388,"end":407,"context":"nd operating it;\r\n- Good English skills (both verbal & writ"}]


### TITLE — CV

In [10]:
preview = examples(audit, label="TITLE", doc_type="cv", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "TITLE cv surfaces")
assert_clean(preview["context"], "TITLE cv bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Project Manager","label":"TITLE","score":0.7755784392,"doc_type":"cv","field":"Position","start":0,"end":15,"context":"Project Manager"},{"surface":"Unity Developer","label":"TITLE","score":0.5841515064,"doc_type":"cv","field":"Position","start":0,"end":15,"context":"Unity Developer"},{"surface":"front-end developer","label":"TITLE","score":0.5902436972,"doc_type":"cv","field":"Looking For","start":30,"end":49,"context":"become an excellent front-end developer, return to IT, and "}]


### TITLE — JD

In [11]:
preview = examples(audit, label="TITLE", doc_type="jd", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "TITLE jd surfaces")
assert_clean(preview["context"], "TITLE jd bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Financial Controller","label":"TITLE","score":0.5856886506,"doc_type":"jd","field":"Long Description","start":21,"end":41,"context":"e are looking for a Financial Controller who will participat"},{"surface":"Financial Controller","label":"TITLE","score":0.6297507882,"doc_type":"jd","field":"Position","start":0,"end":20,"context":"Financial Controller"},{"surface":"iOS Developer","label":"TITLE","score":0.5359680653,"doc_type":"jd","field":"Long Description","start":345,"end":358,"context":"re looking for the  iOS Developer \r\n\r\n\r\n**Responsibil"}]


### DEGREE — CV

In [12]:
preview = examples(audit, label="DEGREE", doc_type="cv", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "DEGREE cv surfaces")
assert_clean(preview["context"], "DEGREE cv bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Exam 483","label":"DEGREE","score":0.6049317718,"doc_type":"cv","field":"Highlights","start":25,"end":33,"context":"soft certification: Exam 483 - Programming in C#"},{"surface":"MCSA","label":"DEGREE","score":0.5087682009,"doc_type":"cv","field":"Highlights","start":56,"end":60,"context":"Programming in C#.\r\nMCSA: Web Applications\r\n"},{"surface":"Master of the self-education","label":"DEGREE","score":0.6263945103,"doc_type":"cv","field":"Highlights","start":151,"end":179,"context":"sting workflow. \r\n- Master of the self-education\r\n- Experience in pe"}]


### DEGREE — JD

In [13]:
preview = examples(audit, label="DEGREE", doc_type="jd", limit=3, context_chars=40)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "DEGREE jd surfaces")
assert_clean(preview["context"], "DEGREE jd bounded context")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Upper-Intermediate+ level in English","label":"DEGREE","score":0.5595124364,"doc_type":"jd","field":"Long Description","start":1500,"end":1536,"context":"r feedback process\r\nUpper-Intermediate+ level in English\r\nFlexibility, desir"},{"surface":"Bachelor's Degree in Computer Science","label":"DEGREE","score":0.7066734433,"doc_type":"jd","field":"Long Description","start":1715,"end":1752,"context":" to no supervision\r\nBachelor's Degree in Computer Science\r\n\r\nYOU WANT TO WORK"},{"surface":"Sc. in Computer Science or Software Engineering","label":"DEGREE","score":0.7263218164,"doc_type":"jd","field":"Long Description","start":1302,"end":1349,"context":"\n\r\nQUALIFICATIONS:\r\nSc. in Computer Science or Software Engineering;\r\n3-7 years of rece"}]


## Semantic questions and rule disagreement

A source-verified span can still have the wrong label, ambiguous boundaries or misleading screening context. Degree-like language about English level or self-education deserves inspection; job-role phrases may appear in aspirations or requirements rather than applied employment history.

An exact rule/model disagreement agrees or differs on source, document, field, offsets and label. Rules are a comparator, not gold. Rule-only spans are not automatically neural false negatives, and neural-only spans are not automatically rule misses. Context categories also remain heuristic. Suspicious-case selection is an inspection heuristic, not an accuracy metric.

### Heuristically flagged title/degree cases

In [14]:
preview = suspicious_examples(audit, limit=3)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "inspection surfaces")
assert_clean(preview["context"], "inspection bounded contexts")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Strong Team Members","label":"TITLE","score":0.5480641723,"doc_type":"cv","field":"Looking For","start":84,"end":103,"context":"ment on Swift using latest technologies.\r\nc) Strong Team Members\r\nd) Professional project manage"},{"surface":"Team Leading","label":"TITLE","score":0.5651910901,"doc_type":"cv","field":"Highlights","start":21,"end":33,"context":"More then 5 years in Team Leading.\r\nExperience in High Load like betting syste"},{"surface":"Upper-Intermediate+ level in English","label":"DEGREE","score":0.5595124364,"doc_type":"jd","field":"Long Description","start":1500,"end":1536,"context":"ly participate in the user feedback process\r\nUpper-Intermediate+ level in English\r\nFlexibility, desire and ability to adapt to"}]

### Shared with current rules

In [15]:
preview = comparison_examples(audit, category="shared", limit=3)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "inspection surfaces")
assert_clean(preview["context"], "inspection bounded contexts")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"bachelor's degree","label":"DEGREE","score":0.623119235,"doc_type":"cv","field":"Moreinfo","start":80,"end":97,"context":"oper, with 2-year experience, graduated with bachelor's degree in Computer Science from the University of D"},{"surface":"Ph.D.","label":"DEGREE","score":0.6703011394,"doc_type":"cv","field":"Highlights","start":165,"end":170,"context":" mobile application of the city portal. \r\nd) Ph.D. in Computer Science"},{"surface":"React","label":"KNOWLEDGE","score":0.9319278598,"doc_type":"cv","field":"Moreinfo","start":108,"end":113,"context":"in Kyiv.\r\nTechnology stack: HTML5, CSS3, JS, React, React Router, i18next, Helmet, AWS S3, AWS "}]


### Only in current rules

In [16]:
preview = comparison_examples(audit, category="rule_only", limit=3)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "inspection surfaces")
assert_clean(preview["context"], "inspection bounded contexts")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"bachelor's degree","label":"DEGREE","score":null,"doc_type":"cv","field":"Highlights","start":92,"end":109,"context":"uated from Kyiv Polytechnic Institute with a bachelor's degree in computer engineering in 2017. At that tim"},{"surface":"AWS","label":"KNOWLEDGE","score":null,"doc_type":"cv","field":"Moreinfo","start":1262,"end":1265,"context":"th Swagger APIs for Blockchain-API usage \r\n- AWS (DynamoDB, S3 , and RDS) for game backend, a"},{"surface":"AWS","label":"KNOWLEDGE","score":null,"doc_type":"cv","field":"Moreinfo","start":146,"end":149,"context":"3, JS, React, React Router, i18next, Helmet, AWS S3, AWS SES\r\n\r\n- To Do List, which works wit"}]


### Only in saved neural output

In [17]:
preview = comparison_examples(audit, category="neural_only", limit=3)
preview["surface"] = preview["surface"].map(redact_text)
preview["context"] = preview["context"].map(redact_text)
assert_clean(preview["surface"], "inspection surfaces")
assert_clean(preview["context"], "inspection bounded contexts")
rendered = preview.to_json(orient="records", force_ascii=False)
assert len(preview) <= 5 and len(rendered) <= 1990
print(rendered)


[{"surface":"Exam 483","label":"DEGREE","score":0.6049317718,"doc_type":"cv","field":"Highlights","start":25,"end":33,"context":"Microsoft certification: Exam 483 - Programming in C#.\r\nMCSA: Web Applications"},{"surface":"MCSA","label":"DEGREE","score":0.5087682009,"doc_type":"cv","field":"Highlights","start":56,"end":60,"context":"ertification: Exam 483 - Programming in C#.\r\nMCSA: Web Applications\r\nMicrosoft Certified: Azur"},{"surface":"Master of the self-education","label":"DEGREE","score":0.6263945103,"doc_type":"cv","field":"Highlights","start":151,"end":179,"context":"lly working automation testing workflow. \r\n- Master of the self-education\r\n- Experience in performing in front of over"}]


## Gold inventory and attribution limits

Current filesystem availability is checked afresh, rather than copied from the historical run's old availability entries. Existing supervised-tagger records belong to their own models and datasets; they must never be quoted as GLiNER accuracy.

GLiNER precision/recall/F1 requires compatible human-labelled spans and saved frozen-configuration predictions on those same gold units. A DataTurks/SkillSpan file alone does not provide predictions for this Djinni run, and any partial label mapping must preserve its boundaries and coverage. No inference on gold sources is performed by this notebook.

In [18]:
gold = inventory_gold()
show_summary(gold)


{
  "sources": {
    "dataturks": {
      "catalog_id": "B1",
      "status": "missing",
      "missing_artifacts": [
        "traindata.json",
        "testdata.json"
      ]
    },
    "skillspan": {
      "catalog_id": "B2",
      "status": "missing",
      "missing_artifacts": [
        "*-train-*.parquet",
        "*-validation-*.parquet",
        "*-test-*.parquet"
      ]
    }
  },
  "existing_record_paths": [
    "extraction/extraction-metrics.json"
  ],
  "gliner_gold_accuracy": "unavailable: no matching frozen-GLiNER gold predictions scored here",
  "warning": "Other learned-tagger records do not measure this saved GLiNER configuration"
}


## Interpretation

Use the source audit to assess whether saved evidence is traceable and complete. Use bounded examples to identify questions for later semantic evaluation. Neither source validity nor a high raw model score establishes a correct label, calibrated confidence, extraction recall or candidate fit.

Missing matching gold/predictions leave semantic accuracy **unavailable**, never zero and never inferred from agreement with seed rules. Supervised extraction evaluation and any downstream ranking benefit remain separate tasks. No final-set label is read, and no screening system is fitted or tuned here.